# Day 11: Text Embeddings and Cosine Similarity

Welcome to Day 11 of the AI Engineering Mastery program. Today, we transition from pure text generation into how machines actually *understand* text: **Embeddings**. 

In this module, we will explore:
1. **Core Theory**: What are embeddings and why do we need them? How do we measure similarity?
2. **Code Implementation**: Generating embeddings using `sentence-transformers` and computing cosine similarity.
3. **Common Pitfalls**: What breaks in production when working with embeddings.
4. **Practical Lab**: An actionable task to build your own semantic search function.


## 1. Core Theory (Just-in-Time)

### The "Why": Machines Don't Read Text
Language Models and search systems cannot fundamentally process strings of text like "apple" or "I love programming." They require mathematical representations of these strings. An **embedding** is a translation of text into a high-dimensional vector (a list of floating-point numbers) where the geometry of the vectors captures the semantic meaning of the text.

If the embedding model is trained well, vectors for "car" and "automobile" will be located close to each other in this high-dimensional space, while "banana" will be far away.

### The "How": Sentence Transformers
While Large Language Models (LLMs) like GPT-4 can generate embeddings, it is often more cost-effective and faster in production to use specialized, smaller models for this task. `sentence-transformers` is an industry-standard Python library built on top of PyTorch and Hugging Face Transformers. It provides access to models like `all-MiniLM-L6-v2`, which are optimized for generating sentence and paragraph embeddings quickly.

### Measuring Distance: Cosine Similarity
Once we have vectors, we need a way to compare them. **Cosine Similarity** measures the cosine of the angle between two non-zero vectors. 
- A similarity of **1.0** means the vectors point in the exact same direction (highly similar semantics).
- A similarity of **0.0** means they are orthogonal (unrelated).
- A similarity of **-1.0** means they are exactly opposite.

Cosine similarity is preferred over Euclidean distance for embeddings because it cares about the *angle* (semantic content) rather than the *magnitude* (length of the text).


## 2. Code Implementation

Let's implement a production-grade class to generate embeddings and compare them. We will use strict type hinting and docstrings.

> **Note on Dependencies**: Make sure you have `sentence-transformers` and `numpy` installed.
> `pip install sentence-transformers numpy`


In [ ]:
import numpy as np
from typing import List, Union
from sentence_transformers import SentenceTransformer, util

class EmbeddingService:
    """
    A service class for generating text embeddings and computing similarities
    using the sentence-transformers library.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2") -> None:
        """
        Initializes the EmbeddingService with a specified model.
        
        Args:
            model_name (str): The Hugging Face model identifier for sentence-transformers.
                              Defaults to 'all-MiniLM-L6-v2', a fast and efficient model.
        """
        # Initialize the model once during class instantiation
        self.model = SentenceTransformer(model_name)

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates dense vector representations (embeddings) for a list of texts.
        
        Args:
            texts (List[str]): A list of strings to embed.
            
        Returns:
            np.ndarray: A 2D numpy array where each row is the embedding for the corresponding text.
        """
        if not texts:
            raise ValueError("The 'texts' list cannot be empty.")
            
        # encode() returns a numpy array by default or a PyTorch tensor if specified.
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        return embeddings

    def compute_cosine_similarity(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        """
        Computes the cosine similarity between two 1D numpy arrays.
        
        Args:
            vec1 (np.ndarray): The first embedding vector.
            vec2 (np.ndarray): The second embedding vector.
            
        Returns:
            float: The cosine similarity score, typically between -1.0 and 1.0.
        """
        # Compute dot product and magnitudes
        dot_product = np.dot(vec1, vec2)
        norm_a = np.linalg.norm(vec1)
        norm_b = np.linalg.norm(vec2)
        
        if norm_a == 0.0 or norm_b == 0.0:
            return 0.0
            
        similarity = dot_product / (norm_a * norm_b)
        return float(similarity)

# --- Example Usage ---
if __name__ == "__main__":
    # 1. Initialize the service
    embedding_service = EmbeddingService()
    
    # 2. Define some texts
    sentences = [
        "The quick brown fox jumps over the lazy dog.",
        "A fast, dark-colored fox leaps over a sleepy hound.",
        "I need to buy some groceries for dinner tonight.",
        "Artificial Intelligence is transforming software engineering."
    ]
    
    # 3. Generate embeddings
    print("Generating embeddings...")
    embeddings = embedding_service.generate_embeddings(sentences)
    
    print(f"Generated {len(embeddings)} embeddings of dimension {embeddings[0].shape[0]}")
    
    # 4. Compute and display similarities
    print("\nCosine Similarities:")
    
    # Compare sentence 0 and 1 (Semantically similar)
    sim_0_1 = embedding_service.compute_cosine_similarity(embeddings[0], embeddings[1])
    print(f"'{sentences[0]}'\nvs\n'{sentences[1]}'\n-> Similarity: {sim_0_1:.4f}\n")
    
    # Compare sentence 0 and 2 (Unrelated)
    sim_0_2 = embedding_service.compute_cosine_similarity(embeddings[0], embeddings[2])
    print(f"'{sentences[0]}'\nvs\n'{sentences[2]}'\n-> Similarity: {sim_0_2:.4f}\n")


## 3. Common Pitfalls

When moving embedding workloads to production, several issues frequently arise:

1. **Model Mismatch**: If you index data in a vector database using `all-MiniLM-L6-v2` but later query it using embeddings generated by `text-embedding-3-small` (OpenAI), the vector spaces are completely incompatible. Searches will return garbage. **Rule**: Always use the exact same model for generating query embeddings that you used for the document embeddings.
2. **Dimension Mismatch**: Different models output vectors of different lengths (e.g., 384 for MiniLM, 1536 for OpenAI). If your vector database (like Qdrant) is configured for 384 dimensions, inserting a 1536-dimensional vector will throw an error.
3. **Re-initializing the Model in a Loop**: Calling `SentenceTransformer('model_name')` inside an API endpoint or a loop will load the heavy model weights from disk into memory every single time, destroying latency. Always initialize the model once globally or within a class constructor (as demonstrated above).
4. **Context Length Truncation**: Small models often have a maximum sequence length (e.g., 256 or 512 tokens). If you pass a 5,000-word essay into `model.encode()`, it will silently truncate the text and only embed the first paragraph. You must chunk long documents before embedding them.


## 4. Practical Lab / Homework

**Your Task:** Build a Semantic Search function.

Using the `EmbeddingService` class defined above, implement a function that takes a query string and a list of document strings. The function should return the top `k` most similar documents based on cosine similarity.

### Requirements:
1. Implement the `semantic_search` function.
2. Ensure you include strict type hinting and a docstring.
3. Do not use stubs or `pass`; provide a fully working implementation.
4. Test it with the provided query and corpus.


In [ ]:
from typing import List, Tuple

def semantic_search(query: str, corpus: List[str], top_k: int = 2) -> List[Tuple[str, float]]:
    """
    Performs a semantic search to find the most relevant documents in a corpus for a given query.
    
    Args:
        query (str): The search query.
        corpus (List[str]): A list of document strings to search through.
        top_k (int): The number of top results to return.
        
    Returns:
        List[Tuple[str, float]]: A list of tuples, where each tuple contains the document string
                                 and its similarity score, sorted in descending order of similarity.
    """
    service = EmbeddingService()
    
    # 1. Embed the query (returns a 2D array, we need the 1st row)
    query_embedding = service.generate_embeddings([query])[0]
    
    # 2. Embed the corpus
    corpus_embeddings = service.generate_embeddings(corpus)
    
    # 3. Compute similarities
    results = []
    for i, doc_embedding in enumerate(corpus_embeddings):
        sim = service.compute_cosine_similarity(query_embedding, doc_embedding)
        results.append((corpus[i], sim))
        
    # 4. Sort results by similarity score in descending order
    results.sort(key=lambda x: x[1], reverse=True)
    
    # 5. Return top_k
    return results[:top_k]

# --- Test your implementation ---
if __name__ == "__main__":
    search_corpus = [
        "Python is a high-level programming language.",
        "The recipe for a perfect chocolate cake requires high-quality cocoa.",
        "Machine learning models require large amounts of data to train.",
        "Baking bread is an art that requires patience and precise measurements.",
        "Vector databases are designed to store and query high-dimensional embeddings."
    ]
    
    search_query = "What do I need to bake a delicious dessert?"
    
    print(f"Query: '{search_query}'\n")
    top_results = semantic_search(search_query, search_corpus, top_k=2)
    
    print("Top Results:")
    for doc, score in top_results:
        print(f"Score: {score:.4f} | Document: '{doc}'")
